<a href="https://colab.research.google.com/github/AsalAbbasNejad/AsalAbbasNejad/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

SR = 16000
DURATION = 1
SAMPLES = SR * DURATION
N_MFCC = 13


def load_and_clean_data(file_name):
    df = pd.read_csv(file_name)

    audio_data = df['Value'].astype(str).str.extract(r'(-?\d+)').astype(float).fillna(0).values.flatten()

    segments = []
    for i in range(0, len(audio_data) - SAMPLES, SAMPLES // 2):
        segment = audio_data[i:i + SAMPLES]

        mfcc = librosa.feature.mfcc(y=segment, sr=SR, n_mfcc=N_MFCC)
        segments.append(mfcc)
    return np.array(segments)

print("Loading data...")
claps = load_and_clean_data('clap.csv')
snaps = load_and_clean_data('snap.csv')
taps = load_and_clean_data('tap.csv')

X = np.concatenate([claps, snaps, taps])
y = np.concatenate([np.zeros(len(claps)), np.ones(len(snaps)), np.zeros(len(taps)) + 2])


X = X[..., np.newaxis]


model = models.Sequential([
    layers.Conv2D(16, (3, 3), activation='relu', input_shape=(X.shape[1], X.shape[2], 1), padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(3, activation='softmax')

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Training model...")
model.fit(X, y, epochs=50, batch_size=8, validation_split=0.2)


converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("model.h", "w") as f:
    f.write("unsigned char model[] = {\n")
    for i, byte in enumerate(tflite_model):
        f.write(f"0x{byte:02x}, " if i < len(tflite_model)-1 else f"0x{byte:02x}")
        if (i + 1) % 12 == 0: f.write("\n")
    f.write("\n};\n")
    f.write(f"unsigned int model_len = {len(tflite_model)};")

print("Finished! Download 'model.h' from the files tab.")

Loading data...
Training model...
Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.3333 - loss: 110.3721 - val_accuracy: 0.0000e+00 - val_loss: 102.8152
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - accuracy: 0.6667 - loss: 38.2548 - val_accuracy: 0.0000e+00 - val_loss: 100.4717
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - accuracy: 0.6667 - loss: 41.7712 - val_accuracy: 0.0000e+00 - val_loss: 98.8848
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - accuracy: 0.3333 - loss: 37.9574 - val_accuracy: 0.0000e+00 - val_loss: 106.2992
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 0.6667 - loss: 10.3937 - val_accuracy: 0.0000e+00 - val_loss: 112.9522
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 0.6667 - loss: 0.7587 - val_accuracy: 0.0000e+00 - val_loss: 130.2449
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step - accuracy: 0.0000e+00 - loss: 75.8232 - val_accuracy: 0.0000e+00 - val_loss: 137.0079
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - accuracy: 0.6667 - loss: 19.